# RFM Customer Segmentation Analysis
**Dataset:** Online Retail (~540,000 rows)  
**Columns:** InvoiceNo, StockCode, Description, Quantity, InvoiceDate, UnitPrice, CustomerID, Country  
**Goal:** Segment customers into High-Value, At-Risk, Low Engagement using RFM + K-Means clustering

## 0. Install dependencies

In [ ]:
!pip install scikit-learn --quiet

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

print('All imports successful.')

## 2. Load & clean data

Key fixes applied:
- `encoding='utf-8-sig'` strips the BOM (`ï»¿`) that appears when reading Excel-exported CSVs with `ISO-8859-1`
- Explicit `format='%d/%m/%Y %H:%M'` for day-first date strings (e.g. `01/12/2010 8:26`)
- `errors='coerce'` converts any unparseable dates to `NaT` instead of crashing

In [ ]:
# ── Load ──────────────────────────────────────────────────────────────────────
df = pd.read_csv('online_retail.csv', encoding='utf-8-sig')

print('Raw shape:', df.shape)
print('Columns:', df.columns.tolist())

In [ ]:
# ── Parse dates (day-first format) ────────────────────────────────────────────
df['InvoiceDate'] = pd.to_datetime(
    df['InvoiceDate'],
    format='%d/%m/%Y %H:%M',
    errors='coerce'
)

bad_dates = df['InvoiceDate'].isna().sum()
print(f'Unparseable dates: {bad_dates}')
df = df.dropna(subset=['InvoiceDate'])

print('Date range:', df['InvoiceDate'].min(), '→', df['InvoiceDate'].max())
print('dtype:', df['InvoiceDate'].dtype)

In [ ]:
# ── Filter & engineer ─────────────────────────────────────────────────────────
df = df.dropna(subset=['CustomerID'])        # remove guest checkouts
df = df[df['Quantity']  > 0]                 # remove cancellations / returns
df = df[df['UnitPrice'] > 0]                 # remove zero-price adjustments

df['Revenue'] = df['Quantity'] * df['UnitPrice']

print('Clean shape:', df.shape)
df.head(3)

## 3. Calculate R, F, M values

In [ ]:
# snapshot_date = one day after the last invoice → acts as "today"
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)
print('Snapshot date:', snapshot_date)

rfm = df.groupby('CustomerID').agg(
    Recency   = ('InvoiceDate', lambda x: (snapshot_date - x.max()).days),
    Frequency = ('InvoiceNo',   'nunique'),
    Monetary  = ('Revenue',     'sum')
).reset_index()

print(f'Unique customers: {len(rfm):,}')
rfm.describe().round(2)

## 4. Score each dimension (1–4 quartiles)

Fix applied: the Online Retail dataset has a heavily skewed `Frequency` distribution (many one-time buyers), causing duplicate quartile bin edges. The helper function below uses `duplicates='drop'` and a rank-based fallback to handle this gracefully.

In [ ]:
def rfm_score(series, ascending=True, n=4):
    """
    Score a series into up to n quartile bins.
    Handles duplicate bin edges (common with skewed distributions)
    via duplicates='drop' and a rank-based fallback.
    """
    labels_asc  = list(range(1, n + 1))
    labels_desc = list(range(n, 0, -1))
    labels = labels_asc if ascending else labels_desc
    try:
        return pd.qcut(
            series,
            q=n,
            labels=labels,
            duplicates='drop'
        ).astype(int)
    except ValueError:
        # Fallback: rank-based scoring for extreme skew
        ranked = series.rank(method='first', ascending=ascending)
        return pd.qcut(ranked, q=n, labels=labels_asc, duplicates='drop').astype(int)


# Recency: lower days = more recent = better → descending
rfm['R_Score'] = rfm_score(rfm['Recency'],   ascending=False)
rfm['F_Score'] = rfm_score(rfm['Frequency'], ascending=True)
rfm['M_Score'] = rfm_score(rfm['Monetary'],  ascending=True)

rfm['RFM_Score'] = rfm['R_Score'] + rfm['F_Score'] + rfm['M_Score']

# Verify: no NaNs, scores in 1–4 range
print('NaNs in scores:', rfm[['R_Score','F_Score','M_Score']].isna().sum().sum())
print('\nScore distributions:')
for col in ['R_Score','F_Score','M_Score']:
    print(f'  {col}:', rfm[col].value_counts().sort_index().to_dict())

## 5. K-Means clustering

### 5a. Elbow method — find optimal k

In [ ]:
scaler     = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm[['R_Score','F_Score','M_Score']])

inertias = []
k_range  = range(2, 9)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(rfm_scaled)
    inertias.append(km.inertia_)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(k_range, inertias, marker='o', color='steelblue', linewidth=2)
ax.set_xlabel('Number of clusters (k)')
ax.set_ylabel('Inertia')
ax.set_title('Elbow method — optimal k selection')
ax.xaxis.set_major_locator(mticker.MultipleLocator(1))
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('elbow_plot.png', dpi=150)
plt.show()
print('Elbow plot saved → elbow_plot.png')

### 5b. Fit K-Means and assign segment labels

Fix applied: replaced `fillna(..., inplace=True)` with a simple reassignment to avoid the pandas Copy-on-Write `ChainedAssignmentError` in pandas 2.0+.

In [ ]:
K = 3   # adjust based on elbow plot

km = KMeans(n_clusters=K, random_state=42, n_init=10)
rfm['Cluster'] = km.fit_predict(rfm_scaled)

# Inspect cluster centroids to assign meaningful labels
cluster_summary = rfm.groupby('Cluster')[['Recency','Frequency','Monetary']].mean()
print('Cluster centroids:\n', cluster_summary.round(2))

# Map clusters to business labels based on centroid characteristics
label_map = {
    cluster_summary['Monetary'].idxmax():  'High-Value',
    cluster_summary['Recency'].idxmax():   'At-Risk',       # highest recency = longest ago
    cluster_summary['Frequency'].idxmin(): 'Low Engagement'
}

# ✅ Reassign instead of inplace=True  (fixes ChainedAssignmentError)
rfm['Segment'] = rfm['Cluster'].map(label_map)
rfm['Segment'] = rfm['Segment'].fillna('Mid-Tier')

print('\nSegment counts:')
print(rfm['Segment'].value_counts())

### 5c. Verify segment characteristics make business sense

In [ ]:
segment_profile = rfm.groupby('Segment')[['Recency','Frequency','Monetary']].mean().round(2)
print(segment_profile)

# Expected pattern:
#   High-Value    → low Recency, high Frequency, high Monetary
#   At-Risk       → high Recency (bought long ago), moderate Monetary
#   Low Engagement→ high Recency, low Frequency, low Monetary

## 6. Visualisations

In [ ]:
COLORS = {
    'High-Value':     '#2196A6',
    'At-Risk':        '#E8913A',
    'Low Engagement': '#C0504D',
    'Mid-Tier':       '#7F7F7F'
}

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('RFM Customer Segmentation', fontsize=14, fontweight='bold')

# ── Chart 1: Customer count per segment ──────────────────────────────────────
counts = rfm['Segment'].value_counts()
axes[0].bar(counts.index, counts.values,
            color=[COLORS.get(s, '#999') for s in counts.index])
axes[0].set_title('Customers per segment')
axes[0].set_ylabel('Number of customers')
axes[0].tick_params(axis='x', rotation=15)

# ── Chart 2: Average monetary value per segment ───────────────────────────────
avg_monetary = rfm.groupby('Segment')['Monetary'].mean().sort_values(ascending=False)
axes[1].bar(avg_monetary.index, avg_monetary.values,
            color=[COLORS.get(s, '#999') for s in avg_monetary.index])
axes[1].set_title('Avg revenue per customer')
axes[1].set_ylabel('Average monetary value (£)')
axes[1].tick_params(axis='x', rotation=15)

# ── Chart 3: Recency vs Monetary scatter ─────────────────────────────────────
for seg, grp in rfm.groupby('Segment'):
    axes[2].scatter(
        grp['Recency'], grp['Monetary'],
        label=seg, alpha=0.4, s=10,
        color=COLORS.get(seg, '#999')
    )
axes[2].set_title('Recency vs Monetary')
axes[2].set_xlabel('Recency (days since last purchase)')
axes[2].set_ylabel('Total spend (£)')
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.savefig('rfm_charts.png', dpi=150)
plt.show()
print('Charts saved → rfm_charts.png')

## 7. Export for Tableau

In [ ]:
# Customer-level summary (for scatter plots, segment filters in Tableau)
rfm.to_csv('rfm_summary.csv', index=False)

# Transaction-level with segment attached (for revenue drill-downs)
df_final = df.merge(
    rfm[['CustomerID','Recency','Frequency','Monetary',
         'R_Score','F_Score','M_Score','RFM_Score','Segment']],
    on='CustomerID',
    how='left'
)
df_final.to_csv('rfm_segmented.csv', index=False)

print(f'rfm_summary.csv   → {len(rfm):,} rows (one per customer)')
print(f'rfm_segmented.csv → {len(df_final):,} rows (one per transaction)')

## 8. Business recommendations

| Segment | Characteristics | Recommended actions |
|---|---|---|
| **High-Value** | Low recency · High frequency · High spend | Loyalty programme, early access, dedicated account manager |
| **At-Risk** | High recency · Previously high spend | Win-back email sequence with discount, re-engagement survey |
| **Low Engagement** | High recency · Low frequency · Low spend | Cost-efficient automated nurture only; sunset flow after 3 no-responses |
| **Mid-Tier** | Moderate across all dimensions | Upsell campaigns, cross-sell recommendations to grow toward High-Value |

### Tableau dashboard checklist
Connect to `rfm_summary.csv` and build four sheets:
1. **Bar chart** — Segment → COUNT(CustomerID)
2. **Treemap** — Segment → SUM(Monetary)
3. **Scatter plot** — Recency (x) × Monetary (y), color by Segment
4. **Customer table** — CustomerID, R/F/M scores, Segment, Country

Use **Segment** as a global dashboard filter action so clicking any segment cross-filters all four views.